# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, following the Croissant schema specification.

### Dataset Source
The dataset source is specified as a Croissant schema and is accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data Loading
Here, we load the dataset metadata and verify access to its fields and record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# If you want to see publication details:
print(f"Citation: {metadata.cite_as}")

## 2. Data Overview
Let's list all available record sets and their associated fields in this dataset, referencing their `@id` where possible.

In [ ]:
# List all record sets, their @id and field @ids
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in this dataset schema.")
else:
    for rs in record_sets:
        print(f"RecordSet: name={rs.name}, @id={rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id={field.id})")
        print("  Columns:")
        for col in rs.columns:
            print(f"    - {col.name} (@id={col.id})")
        print()
# Print a warning if no record sets exist:

## 3. Data Extraction
In this section, we'll attempt to load data from a specific record set (referenced by its `@id`) into a pandas DataFrame for exploration.

> **Note:** If there are no record sets in the dataset, you'll see a message explaining that extraction is not possible.

In [ ]:
dataframes = {}

# Retrieve all record set @ids
record_sets = dataset.record_sets
record_set_ids = [rs.id for rs in record_sets] if record_sets else []

if not record_set_ids:
    print("No record sets are defined in the dataset; data extraction not possible.")
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded data for RecordSet @id {record_set_id} with shape {df.shape}.")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    # Show detailed columns and head for the first record set if available
    if record_set_ids:
        main_record_set_id = record_set_ids[0]
        print(f"First RecordSet DataFrame columns: {dataframes[main_record_set_id].columns.tolist()}")
        display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll now apply basic data wrangling and EDA steps such as filtering, normalization, and grouping to one of the numeric fields, using the `@id` for field references. If there are no record sets or numeric fields, a message will be shown.

In [ ]:
import numpy as np

# EDA on the first record set and first numeric field (if available)
if not record_set_ids:
    print("No record sets, skipping EDA.")
else:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]

    # Attempt to auto-detect a numeric field by pandas dtypes
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        print(f"No numeric fields in RecordSet {main_record_set_id}; skipping numeric EDA.")
    else:
        numeric_field = numeric_candidates[0]
        print(f"Using field '{numeric_field}' for numeric analysis (from @id list)")

        # Filtering
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Attempt to auto-detect a categorical (group) field
        possible_groups = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])]
        if possible_groups:
            group_field = possible_groups[0]
            print(f"Grouping by field '{group_field}' (from @id list)")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped averages by {group_field} for field {numeric_field}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping/categorical fields were found for grouping.")

## 5. Visualization
Let's visualize data distributions and numeric relationships (such as histograms or scatter plots) using matplotlib. This section tries to plot the distribution of the first numeric field (referenced by its `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids:
    print("No data to plot.")
else:
    df = dataframes[main_record_set_id]
    if not numeric_candidates:
        print("No numeric fields to plot.")
    else:
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), bins=20, kde=True, ax=ax)
        ax.set_title(f"Distribution of {numeric_field} (@id)")
        ax.set_xlabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, you:
- Loaded and inspected metadata of the FAIR^2 rangeland management adoption dataset using the Croissant schema and `mlcroissant`.
- Explored the structure (record sets, fields, columns, all by their `@id` values).
- (If available) Extracted tabular records, conducted simple exploratory analysis and normalization, and visualized data distributions.

This approach demonstrates systematic, reproducible data exploration using FAIR metadata standards and programmatic access provided by Croissant and `mlcroissant`.

*Adapt, extend, and refine the EDA for your specific scientific or policy use case!*